# Notebook 22d — Wu 2003: S-B3 pooled bank aggregation (mixed-wide + eta_col-widened top-up)

**Not executed. Depends on notebooks 22b (mixed-wide branch) and 22c (eta_col-widened top-up) having
already produced their bank files.**

Concatenates notebook 22b's original mixed-wide bank (`alpha, beta_r` uniform; `eta_col, xi_reb`
lognormal -- 5000/120/400 rows) with notebook 22c's top-up bank (`alpha, beta_r, eta_col` uniform;
`xi_reb` lognormal -- 2000/60/150 rows) into three combined `.npz` files (train / SBC reduced / SBC
confirm), each row tagged with a `proposal_kind` field (`0` = old 2-dim-wide prior, `1` = new 3-dim-wide
prior) identifying which distribution it was actually drawn from.

**This tag is not cosmetic -- it is required for notebook 24c's pooled-multiproposal training to be
mathematically valid.** `SNPE`'s atomic proposal-correction machinery (the same mechanism already used
in this project's `scripts/`/notebook-48-style pooling) needs to know each row's *true* generating
distribution to correctly re-weight it toward whatever prior is declared at training time. Simply
concatenating the two banks and pretending they are i.i.d. from one distribution would be a real,
silent correctness bug -- exactly the kind of prior/data mismatch already diagnosed once in this
project's own history (the discrete scenario-and-replicate bank declared as if it were a continuous
`box_uniform_4d_sb3()` sample).

In [ ]:
import pathlib, sys
import numpy as np

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

from cstr_sbi.recycle.summaries import N_SUMMARIES_SB

DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

# Sources -- each entry's `uniform_idx` records the EXACT prior distribution that bank was drawn
# from (kind=0 -> mixed_wide_prior_4d_sb3(uniform_idx=(0,1)), kind=1 -> uniform_idx=(0,1,2)).
# Extend this registry if more top-up banks are generated later, mirroring notebook 48's
# BANK_REGISTRY pattern.
SOURCES = {
    'train': [
        {'path': DATA / 'wu2003_sbi_train_sb3_mixedwide_n5000.npz', 'kind': 0, 'uniform_idx': (0, 1), 'label': 'mixedwide_n5000'},
        {'path': DATA / 'wu2003_sbi_train_sb3_etawide_topup_n2000.npz', 'kind': 1, 'uniform_idx': (0, 1, 2), 'label': 'etawide_topup_n2000'},
    ],
    'sbc_reduced': [
        {'path': RESULTS / '24b_sysII_sb3_mixedwide_sbc_set_n120.npz', 'kind': 0, 'uniform_idx': (0, 1), 'label': 'mixedwide_n120'},
        {'path': RESULTS / '24c_sysII_sb3_etawide_topup_sbc_set_n60.npz', 'kind': 1, 'uniform_idx': (0, 1, 2), 'label': 'etawide_topup_n60'},
    ],
    'sbc_confirm': [
        {'path': RESULTS / '24b_sysII_sb3_mixedwide_sbc_set_n400.npz', 'kind': 0, 'uniform_idx': (0, 1), 'label': 'mixedwide_n400'},
        {'path': RESULTS / '24c_sysII_sb3_etawide_topup_sbc_set_n150.npz', 'kind': 1, 'uniform_idx': (0, 1, 2), 'label': 'etawide_topup_n150'},
    ],
}

OUT_PATHS = {
    'train': DATA / 'wu2003_sbi_train_sb3_pooled_etawide_n7000.npz',
    'sbc_reduced': RESULTS / '24c_sysII_sb3_pooled_etawide_sbc_set_n180.npz',
    'sbc_confirm': RESULTS / '24c_sysII_sb3_pooled_etawide_sbc_set_n550.npz',
}
print('OUT_PATHS:', {k: v.name for k, v in OUT_PATHS.items()})

## Section 1 — Load, tag, and concatenate each set type

For each of the three set types (train / SBC reduced / SBC confirm), loads both sources, verifies
they share the same summary dimension, concatenates `thetas4`/`summaries`, and builds the
`proposal_kind` provenance array.

In [ ]:
def load_and_tag(entry):
    if not entry['path'].exists():
        raise FileNotFoundError(
            f"{entry['path'].name} not found. Run notebook 22b (mixed-wide branch) and/or "
            "notebook 22c (eta_col-widened top-up) first."
        )
    data = np.load(entry['path'])
    theta = np.asarray(data['thetas4'], dtype=np.float32)
    summary = np.asarray(data['summaries'], dtype=np.float32)
    kind = np.full(theta.shape[0], entry['kind'], dtype=np.int64)
    print(f"  loaded {entry['label']}: {entry['path'].name} -> {theta.shape[0]} rows, kind={entry['kind']}")
    return theta, summary, kind


aggregated = {}
for set_name, entries in SOURCES.items():
    print(f'=== {set_name} ===')
    thetas, summaries, kinds = [], [], []
    for entry in entries:
        theta, summary, kind = load_and_tag(entry)
        assert summary.shape[1] == N_SUMMARIES_SB
        thetas.append(theta)
        summaries.append(summary)
        kinds.append(kind)
    theta_all = np.concatenate(thetas, axis=0)
    summary_all = np.concatenate(summaries, axis=0)
    kind_all = np.concatenate(kinds, axis=0)
    aggregated[set_name] = (theta_all, summary_all, kind_all)
    print(f'  combined: {theta_all.shape[0]} rows ({np.bincount(kind_all).tolist()} by kind)')
    print()

## Section 2 — Save combined banks with provenance metadata

Each saved file stores `uniform_idx_by_kind` (a small lookup table: `kind -> uniform_idx tuple`) so
notebook 24c can reconstruct the exact `mixed_wide_prior_4d_sb3(uniform_idx=...)` proposal object for
each row's source without hardcoding it a second time.

In [ ]:
UNIFORM_IDX_BY_KIND = {0: (0, 1), 1: (0, 1, 2)}

for set_name, (theta_all, summary_all, kind_all) in aggregated.items():
    out_path = OUT_PATHS[set_name]
    np.savez(
        out_path,
        thetas4=theta_all,
        summaries=summary_all,
        proposal_kind=kind_all,
        uniform_idx_kind0=np.asarray(UNIFORM_IDX_BY_KIND[0], dtype=np.int64),
        uniform_idx_kind1=np.asarray(UNIFORM_IDX_BY_KIND[1], dtype=np.int64),
        structure='S-B3',
        note='Pooled bank: kind=0 from mixed_wide_prior_4d_sb3(uniform_idx=(0,1)), kind=1 from uniform_idx=(0,1,2).',
    )
    print(f'saved {out_path.name}: {theta_all.shape[0]} rows')

## Section 3 — Round-trip verification and acceptance criteria

This branch passes if:
1. each combined set's row count equals the sum of its two sources,
2. all summaries are finite and have the expected `S-B` dimension,
3. `proposal_kind` only takes values `{0, 1}` and both are present in the training set,
4. the two `uniform_idx` groups' `alpha`/`beta_r` ranges agree (both should span the full uniform
   box, since both source priors already widen those two dimensions).

In [ ]:
theta_train_pooled, summary_train_pooled, kind_train_pooled = aggregated['train']

rt = np.load(OUT_PATHS['train'])
criteria = [
    ('train row count = sum of sources', rt['thetas4'].shape[0] == theta_train_pooled.shape[0], f"{rt['thetas4'].shape[0]} vs {theta_train_pooled.shape[0]}"),
    ('all summaries finite', bool(np.isfinite(summary_train_pooled).all()), 'finite check'),
    ('summary dimension is S-B', summary_train_pooled.shape[1] == N_SUMMARIES_SB, str(summary_train_pooled.shape[1])),
    ('proposal_kind only 0/1', bool(set(np.unique(kind_train_pooled).tolist()) <= {0, 1}), np.unique(kind_train_pooled).tolist()),
    ('both kinds present in training set', bool((kind_train_pooled == 0).any() and (kind_train_pooled == 1).any()), np.bincount(kind_train_pooled).tolist()),
]
for name, passed, value in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}: {value}")

print()
print('Next step: notebook 24c trains SNPE on this pooled bank using the atomic proposal-correction')
print('mechanism (one append_simulations call per proposal_kind, each tagged with its own true prior).')